# Module 7 — Cluster Evaluation & Customer Profiling
Reuses the Day-6 engineered dataset and saved production pipeline (`outputs/models/pipeline.pkl`). No Day-6 file is modified; cluster labels are obtained via `.predict()` on the already-fitted model, not by retraining.

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "src"))
from cluster_profiling import *


## 1. Load Project Outputs

In [3]:
df = load_day6_dataset()
pipeline = load_day6_pipeline()
labels = assign_cluster_labels(df, pipeline)

print(f"Dataset: {df.shape}")
print(f"Model: {type(pipeline['model']).__name__}")
print(f"Features used by pipeline: {len(pipeline['features'])}")


Dataset: (2237, 52)
Model: KMeans
Features used by pipeline: 45


/home/xploit/Downloads/Day-6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(


## 2. Dataset Validation

In [4]:
validation = validate_dataset(df, labels)
validation


,Check,Result
0,Shape,2237 rows x 52 columns
1,Missing values,0
2,Duplicate rows,184
3,Cluster count,2
4,Cluster label counts,"{0: np.int64(1049), 1: np.int64(1188)}"


## 3. Cluster Statistics

In [5]:
stats = cluster_statistics(df, labels)
save_report(stats, "cluster_statistics")
stats


,Customer_Count,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,Marital_Status_Widow,Preferred_Shopping_Channel_Catalog,Preferred_Shopping_Channel_Store,Preferred_Shopping_Channel_Web,Product_Preference_Fish,Product_Preference_Fruits,Product_Preference_Gold,Product_Preference_Meat,Product_Preference_Sweets,Product_Preference_Wine
Cluster,,,,,,,,,,,,,,,,,,,,,
0,1049,-0.791348,0.688621,-0.023285,-0.008421,-0.865196,-0.661071,-0.865900,-0.663476,-0.646694,...,0.023832,0.002860,0.734986,0.262154,0.046711,0.014299,0.146806,0.197331,0.013346,0.581506
1,1188,0.698757,-0.608050,0.020561,0.007436,0.763965,0.583724,0.764587,0.585847,0.571029,...,0.043771,0.111953,0.613636,0.274411,0.008418,0.000000,0.010943,0.198653,0.001684,0.780303


## 4. Demographic Analysis
Age, Income, Family_Size, and Total_Children are standardized (z-scored) in the Module 5 engineered dataset. Education/Marital_Status are one-hot encoded, so their values below are cluster proportions.

In [6]:
demo = demographic_report(df, labels)
save_report(demo, "demographic_report")
demo


,Customer_Count,Age,Income,Family_Size,Total_Children,Education_2N Cycle,Education_Basic,Education_Graduation,Education_Master,Education_Phd,Marital_Status_Divorced,Marital_Status_Married,Marital_Status_Single,Marital_Status_Together,Marital_Status_Widow
Cluster,,,,,,,,,,,,,,,
0,1049,-0.203394,-0.791348,0.408716,0.476198,0.104862,0.049571,0.484271,0.167779,0.193518,0.098189,0.393708,0.220210,0.264061,0.023832
1,1188,0.179596,0.698757,-0.360895,-0.420481,0.076599,0.001684,0.521044,0.163300,0.237374,0.107744,0.379630,0.214646,0.254209,0.043771


## 5. Spending Behaviour

In [7]:
spend = spending_report(df, labels)
save_report(spend, "spending_report")
spend


,Customer_Count,Total_Spending,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,Segment_Label
Cluster,,,,,,,,,
0,1049,-0.929795,-0.865196,-0.661071,-0.865900,-0.663476,-0.646694,-0.641349,Budget-Conscious (Lowest Spending)
1,1188,0.821006,0.763965,0.583724,0.764587,0.585847,0.571029,0.566309,Premium (Highest Spending)


## 6. Shopping Behaviour & Engagement

In [8]:
shop = shopping_behavior_report(df, labels)
save_report(shop, "shopping_behavior")
shop


,Customer_Count,NumWebPurchases,NumStorePurchases,NumCatalogPurchases,NumDealsPurchases,Deal_Dependency,NumWebVisitsMonth,Recency,Dominant_Channel,Deal_Seeker,Activity_Status
Cluster,,,,,,,,,,,
0,1049,-0.681454,-0.782318,-0.724052,-0.035906,0.690017,0.497182,-0.008421,NumWebPurchases,True,Active (Lowest Recency)
1,1188,0.601722,0.690784,0.639335,0.031705,-0.609283,-0.439010,0.007436,NumStorePurchases,False,Inactive (Highest Recency)


## 7. Marketing Campaign Analysis

In [9]:
camp = campaign_report(df, labels)
save_report(camp, "campaign_analysis")
camp


,Customer_Count,AcceptedCmp1,AcceptedCmp2,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,Response,Complain,Responsiveness
Cluster,,,,,,,,,
0,1049,0.001907,0.001907,0.070543,0.015253,0.000000,0.095329,0.011439,Marketing-Resistant (Needs Re-engagement)
1,1188,0.119529,0.023569,0.074916,0.127104,0.136364,0.196970,0.006734,Campaign-Responsive


## 8. Business Segment Naming
Names are assigned from the computed stats above, not chosen freehand.

In [10]:
names = name_segments(spend, shop, camp)
save_report(names, "cluster_names")
names


,Segment_Name,Justification
Cluster,,
0,Budget-Conscious Deal Seekers,Spending: Budget-Conscious (Lowest Spending); ...
1,Premium Loyal Customers,Spending: Premium (Highest Spending); dominant...


## 9. Customer Personas
`Estimated_CLV_Proxy` is average historical `Total_Spending` per cluster — a simple, transparent proxy, not a discounted/forecasted CLV model (no such model exists in this project).

In [11]:
personas = build_personas(df, labels, names, demo, spend, shop, camp)
save_report(personas, "customer_personas")
personas


,Persona_Name,Age_Profile,Income_Level,Family_Status,Shopping_Habits,Preferred_Product,Preferred_Channel,Marketing_Responsiveness,Customer_Challenges,Recommended_Marketing_Strategy,Estimated_CLV_Proxy
Cluster,,,,,,,,,,,
0,Budget-Conscious Deal Seekers,Below-average age (standardized mean z=-0.20; ...,-0.79 (standardized mean; higher = above datas...,"Avg family size 0.41, avg children 0.48 (stand...","Dominant channel: NumWebPurchases, activity: A...",Gold,NumWebPurchases,Marketing-Resistant (Needs Re-engagement),Low campaign responsiveness,"Value-driven, deal-based re-engagement campaigns",-0.93 (avg. historical Total_Spending; not a f...
1,Premium Loyal Customers,Above-average age (standardized mean z=0.18; r...,0.70 (standardized mean; higher = above datase...,"Avg family size -0.36, avg children -0.42 (sta...","Dominant channel: NumStorePurchases, activity:...",Meat,NumStorePurchases,Campaign-Responsive,None flagged in available campaign data,Loyalty and retention offers on preferred prod...,0.82 (avg. historical Total_Spending; not a fo...


## 10. Visualizations
PCA and t-SNE plots are reused directly from Day-6 (`outputs/figures/pca.png`, `tsne.png`) rather than regenerated, since the clusters are unchanged.

In [12]:
plot_cluster_distribution(labels, FIGURES_DIR / "cluster_distribution.png")
plot_income_comparison(df, labels, FIGURES_DIR / "income_comparison.png")
plot_spending_comparison(spend, FIGURES_DIR / "spending_comparison.png")
plot_product_heatmap(spend, FIGURES_DIR / "product_heatmap.png")
plot_purchase_channels(shop, FIGURES_DIR / "purchase_channels.png")
plot_campaign_response(camp, FIGURES_DIR / "campaign_response.png")
plot_recency_comparison(df, labels, FIGURES_DIR / "recency_comparison.png")

radar_features = ["Income", "Total_Spending", "Recency", "Age"]
plot_radar(stats, radar_features, FIGURES_DIR / "radar_clusters.png")

reuse_day6_figure("pca.png", FIGURES_DIR / "pca_clusters.png")
reuse_day6_figure("tsne.png", FIGURES_DIR / "tsne_clusters.png")

print("Figures saved to:", FIGURES_DIR)


Figures saved to: /home/xploit/Downloads/ML-Projects/Day-7/outputs/figures


## 11. Segment Comparison

In [13]:
comparison = segment_comparison(names, demo, spend, shop, camp)
save_report(comparison, "segment_comparison")
comparison


,Segment_Name,Avg_Income,Avg_Total_Spending,Avg_Total_Purchases,Engagement_Status,Campaign_Response,Preferred_Channel,Business_Value,Marketing_Strategy
Cluster,,,,,,,,,
0,Budget-Conscious Deal Seekers,-0.791348,-0.929795,-2.223730,Active (Lowest Recency),Marketing-Resistant (Needs Re-engagement),NumWebPurchases,Budget-Conscious (Lowest Spending),"Value-driven, deal-based campaigns"
1,Premium Loyal Customers,0.698757,0.821006,1.963546,Inactive (Highest Recency),Campaign-Responsive,NumStorePurchases,Premium (Highest Spending),Loyalty and retention offers


## 12. Final Export

In [14]:
final_profiles = names.join(demo).join(spend.drop(columns=["Segment_Label", "Customer_Count"])) \
    .join(shop.drop(columns=["Dominant_Channel", "Deal_Seeker", "Activity_Status", "Customer_Count"])) \
    .join(camp.drop(columns=["Responsiveness", "Customer_Count"])) \
    .join(personas[["Age_Profile", "Preferred_Product", "Preferred_Channel",
                     "Marketing_Responsiveness", "Customer_Challenges",
                     "Recommended_Marketing_Strategy", "Estimated_CLV_Proxy"]])

save_report(final_profiles, "final_customer_profiles")
final_profiles.T


Cluster,0,1
Segment_Name,Budget-Conscious Deal Seekers,Premium Loyal Customers
Justification,Spending: Budget-Conscious (Lowest Spending); ...,Spending: Premium (Highest Spending); dominant...
Customer_Count,1049,1188
Age,-0.203394,0.179596
Income,-0.791348,0.698757
Family_Size,0.408716,-0.360895
Total_Children,0.476198,-0.420481
Education_2N Cycle,0.104862,0.076599
Education_Basic,0.049571,0.001684
Education_Graduation,0.484271,0.521044


## Summary

Two clusters (inherited from Day-6's k=2 selection) were profiled across demographics, spending, shopping behavior, and campaign response, using only the saved Day-6 dataset and pipeline — no retraining occurred. All 9 reports are saved to `outputs/reports/` and all 10 figures to `outputs/figures/` (2 of which — PCA and t-SNE — are direct copies of the Day-6 versions).